# Stereo-aware, Protection-agnostic Retrosynthetic Planning

This tutorial plans **Ravidasvir** while deliberately removing target stereochemistry and conservative protecting groups from the structure seen by the tree search. After planning, the winning routes are converted to the JSON-like route representation and passed through SynPlanner's ordered route-postprocessing pipeline.

The workflow is:

1. standardize the protected, stereo-labelled target;
2. remove protecting groups while recording exact preprocessing provenance;
3. remove stereochemical labels from the deprotected planning target;
4. search against the prepared building-block stock using full Standard InChIKeys;
5. export winning routes as JSON;
6. automatically restore the protected target and target stereochemistry, then validate leaf stereochemistry against the BB catalog.

The inserted protection reactions are structural bookkeeping steps. They do not imply reagents, conditions, or experimental feasibility. Full protected-BB leaf expansion is demonstrated as an opt-in setting because the large rerun catalog can produce tens of thousands of Cartesian route variants.


## 1. Set up input and output locations

This notebook uses the existing GPS policy and reaction rules under `synplan_data`. The search stock and its identity catalog come from the requested rerun under `bb_exp/all-bb-2026-06-rerun`.

Run the notebook from the SynPlanner environment. The absolute project path below makes the data selection explicit.

In [ ]:
from pathlib import Path

project_root = Path("/data1/agilmullin/almaz/SynPlanner").resolve()

ranking_policy_network = (
    project_root
    / "synplan_data/policy/supervised_gps/v1/v1/ranking_policy.ckpt"
)
reaction_rules_path = (
    project_root / "synplan_data/policy/supervised_gps/v1/reaction_rules.tsv"
)

bb_root = project_root / "bb_exp/all-bb-2026-06-rerun"
bb_inchikey_file = bb_root / "building_blocks.inchikey"
# bb_smi_file = bb_root / "building_blocks.smi"
bb_identity_file = bb_root / "building_blocks_identity.tsv"

results_folder = project_root / "tutorial_results"
results_folder.mkdir(parents=True, exist_ok=True)

raw_routes_path = results_folder / "ravidasvir_routes_raw.json"
postprocessed_routes_path = (
    results_folder / "ravidasvir_routes_postprocessed.json"
)
postprocess_diagnostics_path = (
    results_folder / "ravidasvir_postprocess_diagnostics.json"
)

print(f"Policy: {ranking_policy_network}")
print(f"Rules: {reaction_rules_path}")
print(f"InChIKey stock: {bb_inchikey_file}")
print(f"Identity catalog: {bb_identity_file}")

## 2. Prepare the Ravidasvir planning target

Ravidasvir contains four tetrahedral stereo labels and two methyl-carbamate protecting groups recognized by the conservative `amine_methylcarbamate` rule.

The protected target is standardized with stereo retained. Deprotection is then performed with provenance capture before stereo is removed. Keeping this order is important: postprocessing needs the protected stereo-labelled structure and the exact protected-to-deprotected transformation, even though the tree searches a stereo-free molecule.

In [ ]:
from synplan.chem.building_blocks import (
    deprotect_molecule_with_provenance,
)
from synplan.chem.utils import mol_from_smiles, safe_canonicalization

ravidasvir_smiles = (
    "COC(=O)N[C@@H](C(C)C)C(=O)N1CCC[C@H]1C1=NC(=CN1)"
    "C1=CC2=C(C=C1)C=C(C=C2)C1=CC2=C(NC(=N2)[C@@H]2CCCN2"
    "C(=O)[C@@H](NC(=O)OC)C(C)C)C=C1"
)

protected_target = mol_from_smiles(
    ravidasvir_smiles,
    standardize=True,
    clean_stereo=False,
    clean2d=True,
)
protected_target_smiles = str(protected_target)

protected_target

In [ ]:
deprotected_target, target_provenance = (
    deprotect_molecule_with_provenance(
        protected_target,
        policy="conservative",
    )
)

planning_target = deprotected_target.copy()
planning_target.clean_stereo()
planning_target = safe_canonicalization(
    planning_target,
    clean_stereo=False,
)
planning_target.clean2d()

planning_target_smiles = str(planning_target)
planning_target


In [ ]:
import json

target_events = (
    json.loads(target_provenance["deprotection_events"])
    if target_provenance is not None
    else []
)

print("Standardized protected target:")
print(protected_target_smiles)
print("\nDeprotected target with stereo retained for provenance:")
print(str(deprotected_target))
print("\nStereo-free, deprotected target used by tree search:")
print(planning_target_smiles)
print("\nRecorded target deprotection events:")
for event in target_events:
    print(
        f"  pass={event['pass_index']} "
        f"rule={event['rule_name']} "
        f"mapping={event['query_mapping']}"
    )

## 3. Load the InChIKey building-block stock and retrosynthesis models

`load_building_block` returns a typed stock. By explicitly requesting `identity_format="inchikey"`, terminal precursor membership is evaluated through full Standard InChIKeys rather than canonical-SMILES string equality.

The larger identity catalog is loaded only after tree search, when protected BB alternatives and stereo validation are needed.

In [ ]:
from synplan.chem.building_blocks import BuildingBlockStockLoadConfig
from synplan.utils.loading import (
    load_building_block,
    load_policy_function,
    load_reaction_rules,
)

building_blocks = load_building_block(
    bb_inchikey_file,
    config=BuildingBlockStockLoadConfig(
        identity_format="inchikey",
    ),
)
# building_blocks = load_building_block(
#     bb_smi_file,
#     config=BuildingBlockStockLoadConfig(
#         identity_format="smiles",
#         standardize=False
#     ),
# )


In [ ]:

reaction_rules = load_reaction_rules(reaction_rules_path)
policy_function = load_policy_function(
    weights_path=ranking_policy_network,
)

print(
    f"Loaded {len(building_blocks):,} unique "
    f"{building_blocks.identity_format} building-block keys"
)
print(f"Loaded {len(reaction_rules):,} reaction rules")

## 4. Configure the tree search

Ravidasvir is substantially more complex than the small examples in the basic planning tutorial, so this example uses a larger depth and search budget. Increase `max_iterations` or `max_time` if no winning route is found on the first run.

No protection-aware route scorer is supplied here: the search intentionally operates on the deprotected, stereo-free target. Protection and stereochemistry are restored only after winning routes have been exported.

In [ ]:
from synplan.utils.config import (
    RolloutEvaluationConfig,
    TreeConfig,
)
from synplan.utils.loading import load_evaluation_function

tree_config = TreeConfig(
    search_strategy="expansion_first",
    max_iterations=300,
    max_time=120,
    max_depth=12,
    min_mol_size=1,
    init_node_value=0.5,
    ucb_type="uct",
    c_ucb=0.1,
)

evaluation_config = RolloutEvaluationConfig(
    policy_network=policy_function,
    reaction_rules=reaction_rules,
    building_blocks=building_blocks,
    min_mol_size=tree_config.min_mol_size,
    max_depth=tree_config.max_depth,
    normalize=True,
)
evaluation_function = load_evaluation_function(
    evaluation_config,
)

In [ ]:
from synplan.mcts.tree import Tree

tree = Tree(
    target=planning_target,
    config=tree_config,
    reaction_rules=reaction_rules,
    building_blocks=building_blocks,
    expansion_function=policy_function,
    evaluation_function=evaluation_function,
)

## 5. Run retrosynthetic planning

Each iteration advances the MCTS search. The tree records every winning node found within the configured time and iteration bounds.

In [ ]:
tree.run()

print(f"Winning routes: {len(tree.winning_nodes)}")
print(f"Winning node IDs: {tree.winning_nodes}")
tree

### Visualize the raw search routes

These routes still contain the deprotected, stereo-free target and any deprotected BB identities used during search. Transparent boxes make the status-colored borders easier to compare.

In [ ]:
from IPython.display import SVG, display

from synplan.utils.visualisation import get_route_svg

for route_number, node_id in enumerate(tree.winning_nodes, start=1):
    print(
        f"-------- Raw route {route_number}: "
        f"winning node #{node_id} --------"
    )
    display(
        SVG(
            get_route_svg(
                tree,
                node_id,
                box_solid=False,
            )
        )
    )
    if route_number == 4:
        break

## 6. Convert winning routes to JSON

`export_tree_to_json` writes the v1 JSON-like route trees and returns the same trees in memory. The exported molecule leaves retain their `in_stock` flags, which the postprocessor uses when expanding protected BB alternatives.

In [ ]:
from synplan.chem.reaction.routes.io import export_tree_to_json

if tree.winning_nodes:
    export_result = export_tree_to_json(
        tree,
        raw_routes_path,
        strict=False,
    )
    routes_json = export_result.routes
    print(
        f"Exported {len(routes_json)} raw routes "
        f"to {raw_routes_path}"
    )
    for diagnostic in export_result.diagnostics:
        print(
            f"Export diagnostic for route {diagnostic.route_id}: "
            f"{diagnostic.message}"
        )
else:
    export_result = None
    routes_json = {}
    print(
        "No winning routes were available for export. "
        "Increase the search budget and rerun from section 5."
    )

## 7. Run ordered route postprocessing

The postprocessing pipeline compares the original standardized target with every route root and automatically infers whether stereo restoration, protection restoration, both, or neither is required. When enabled, the complete order is:

1. restore the protected target while enumerating valid protection-group orders;
2. replace each deprotected in-stock leaf with protected purchasable BB alternatives;
3. restore and propagate target stereochemistry and validate stereo-bearing BB identities;
4. calculate route cost.

> **Stereo-propagation limitation:** this procedure transfers structurally valid labels through atom mapping; it is not reaction-aware stereochemical prediction. It assumes an unchanged configuration when a mapped stereocentre remains valid and does not predict inversion, racemization, epimerization, or newly formed stereocentres. A reaction-affected centre can therefore remain unflagged and requires independent chemical validation. Terminal BB catalog validation does not validate the stereochemical course of intermediate reactions.

Some deprotected leaves in this large rerun catalog have many protected source alternatives, making full route expansion a Cartesian product of tens of thousands of variants. This bounded tutorial therefore disables protected-BB expansion explicitly while still restoring target protection and stereo. Set `expand_deprotected=True` and choose `max_variants_per_route` deliberately when those concrete BB alternatives are required.

The rerun directory contains an InChIKey stock and identity catalog but no prepared self-describing price artifact with `source_index`, so costing is also disabled explicitly.


In [ ]:
from synplan.chem.building_blocks import BuildingBlockCatalog

building_block_catalog = BuildingBlockCatalog.from_files(
    bb_identity_file,
    identity_format="inchikey",
)

print(
    f"Loaded {len(building_block_catalog):,} "
    "catalog identities for postprocessing"
)

In [ ]:
from synplan.chem.reaction.routes.postprocess import RoutePostprocessConfig, postprocess_routes

postprocessed = postprocess_routes(
    routes_json,
    building_block_catalog,
    target_smiles=protected_target_smiles,
    preprocessing_provenance=target_provenance,
    config=RoutePostprocessConfig(expand_deprotected=False, calculate_cost=False),
)


In [ ]:
postprocessed_routes = {
    f"{item.route_id}:{item.variant_index}": item.route
    for item in postprocessed.variants
}
stereo_mismatches = [
    item for item in postprocessed.variants
    if item.route.get("stereo_mismatch", False)
]
print(f"Created {len(postprocessed_routes)} route variants")
print(f"{len(stereo_mismatches)} routes contain a BB stereo mismatch")


### Inspect diagnostics and save postprocessed routes

A diagnostic is local to one source route and one postprocessing stage. For example, a route can be rejected because its target/BB Cartesian product exceeds the configured bound, while other routes still complete.

In [ ]:
from dataclasses import asdict
import json

for diagnostic in postprocessed.diagnostics:
    print(
        f"route={diagnostic.route_id} "
        f"stage={diagnostic.stage} "
        f"type={diagnostic.exception_type} "
        f"message={diagnostic.message}"
    )

postprocessed_routes = {
    f"{item.route_id}:{item.variant_index}": item.route
    for item in postprocessed.variants
}
diagnostic_records = [
    asdict(diagnostic)
    for diagnostic in postprocessed.diagnostics
]

with open(postprocessed_routes_path, "w") as stream:
    json.dump(
        postprocessed_routes,
        stream,
        indent=2,
    )

with open(postprocess_diagnostics_path, "w") as stream:
    json.dump(
        diagnostic_records,
        stream,
        indent=2,
        default=str,
    )

print(f"Postprocessed routes saved to {postprocessed_routes_path}")
print(f"Diagnostics saved to {postprocess_diagnostics_path}")

### Visualize postprocessed route variants

The target at the root now has its original stereochemistry and methyl-carbamate groups. In this bounded example, BB leaves retain the planner identities because full protected-BB expansion was disabled above. Enabling `expand_deprotected` inserts explicit deprotection bookkeeping reactions leading from protected catalog inputs to those leaves.


In [ ]:
from synplan.utils.visualisation import (
    get_route_svg_from_json,
)

for variant_number, route_id in enumerate(
    postprocessed_routes,
    start=1,
):
    print(
        f"-------- Postprocessed variant "
        f"{variant_number}: {route_id} --------"
    )
    display(
        SVG(
            get_route_svg_from_json(
                postprocessed_routes,
                route_id,
                box_solid=False,
            )
        )
    )
    if variant_number == 4:
        break

## 8. Inspect restoration metadata

Target restoration records the sequence mode, rule order, and step count at the route root. Stereo validation records an aggregate `stereo_mismatch` flag and detailed per-leaf catalog comparisons.

With two distinguishable protecting groups, bounded enumeration can produce multiple protection-order variants. Symmetry-equivalent molecular sequences are deduplicated by the postprocessor.

In [ ]:
if postprocessed.variants:
    first_variant = postprocessed.variants[0]
    first_route = first_variant.route

    print(f"Source route ID: {first_variant.route_id}")
    print(f"Variant index: {first_variant.variant_index}")
    print(
        "Target protection restored:",
        first_route.get("target_protection_restored"),
    )
    print(
        "Target protection sequence mode:",
        first_route.get("target_protection_sequence_mode"),
    )
    print(
        "Target protection rule sequence:",
        first_route.get("target_protection_rule_sequence"),
    )
    print(
        "Target protection steps:",
        first_route.get("target_protection_steps"),
    )
    print(
        "Any BB stereo mismatch:",
        first_route.get("stereo_mismatch"),
    )
else:
    print(
        "No postprocessed variant is available yet. "
        "Review the search and postprocessing diagnostics above."
    )

## Notes for larger production runs

- Keep the protected target SMILES and its preprocessing provenance together; the provenance binds restoration to the exact taxonomy and preprocessing transformation.
- Full Standard InChIKeys retain stereochemical identity, while postprocessing additionally reports catalog mismatches at individual BB leaves.
- `max_variants_per_route` bounds the complete Cartesian product of target protection sequences and protected-BB alternatives. Increase it deliberately rather than allowing unbounded expansion.
- To enable final cost calculation, regenerate or provide a prepared price table containing explicit `source_index`, `input_smiles`, and vendor `*_ppg` columns, load it through `BuildingBlockCatalog.from_files(..., price_file=...)`, and set `calculate_cost=True`.